In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Drive
drive.mount('/content/drive')

PROJECT_DIR = Path("/content/drive/MyDrive/AI-ML-Internship/Week-6-LLM-API-Summarizer")
NOTEBOOKS_DIR = PROJECT_DIR / "notebooks"
SRC_DIR = PROJECT_DIR / "src"
OUTPUTS_DIR = PROJECT_DIR / "outputs"

for folder in [NOTEBOOKS_DIR, SRC_DIR, OUTPUTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project Path Ready:", PROJECT_DIR)

In [2]:
# 1. Fix google-auth version conflict
!pip install -q google-auth==2.49.0

# 2. Install Google GenAI SDK
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 19.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.1 which is incompatible.


In [3]:
from google import genai
from google.colab import userdata

# Load API Key from Secrets
API_KEY = userdata.get("GEMINI_API_KEY")
if not API_KEY:
    raise ValueError("GEMINI_API_KEY not found in Colab Secrets.")

# Initialize Client
client = genai.Client(api_key=API_KEY)
print("Gemini client initialized successfully!")

# Test connection
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Explain artificial intelligence in two sentences."
)
print("\nTest Response:\n", response.text)

Gemini client initialized successfully!

Test Response:
 Artificial intelligence is a field of technology that enables computers to mimic human thinking by learning from data, recognizing patterns, and solving complex problems. Over time, these systems use that learned knowledge to make independent decisions, adapt to new information, and perform tasks that once required human intellect.


In [5]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/AI-ML-Internship/Week-6-LLM-API-Summarizer")
NOTEBOOKS_DIR = PROJECT_DIR / "notebooks"
SRC_DIR = PROJECT_DIR / "src"
OUTPUTS_DIR = PROJECT_DIR / "outputs"


for folder in [NOTEBOOKS_DIR, SRC_DIR, OUTPUTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)



summarizer_code = '''import os
from google import genai

MODEL_NAME = "gemini-3.8-flash"

def create_client():
    """Create and return a Gemini API client using environment variables."""
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        raise ValueError("GEMINI_API_KEY environment variable is not set.")
    return genai.Client(api_key=api_key)

def validate_text(text):
    """Validate user-provided text."""
    text = text.strip()
    if not text:
        raise ValueError("Text cannot be empty.")
    if len(text.split()) < 20:
        raise ValueError("Please provide at least 20 words for summarization.")
    return text

def summarize_text(client, text):
    """Send text to Gemini and return a 3-point summary with prompt engineering."""
    prompt = f\"\"\"
You are a professional text summarization assistant.

Your task is to summarize the provided text.

Rules:
1. Produce exactly 3 bullet points.
2. Each bullet should contain one important idea.
3. Keep each bullet concise and informative.
4. Do not add information that is not present in the text.
5. Return only the 3 bullet points.
6. Do not include an introduction or conclusion.

TEXT:{text}
\"\"\"
    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )
        if not response.text:
            raise RuntimeError("The Gemini API returned an empty response.")
        return response.text
    except Exception as error:
        return f"API Error: {error}"

def main():
    print("=" * 60)
    print("           GEMINI AI TEXT SUMMARIZER")
    print("=" * 60)

    try:
        client = create_client()
        text = input("\\nEnter a long paragraph to summarize:\\n\\n")
        text = validate_text(text)

        print("\\nGenerating summary...\\n")
        summary = summarize_text(client, text)

        print("=" * 60)
        print("SUMMARY")
        print("=" * 60)
        print(summary)

    except ValueError as error:
        print(f"\\nInput/Configuration Error: {error}")
    except Exception as error:
        print(f"\\nUnexpected Error: {error}")

if __name__ == "__main__":
    main()
'''

script_path = SRC_DIR / "summarizer.py"
with open(script_path, "w", encoding="utf-8") as f:
    f.write(summarizer_code)

print(f"Successfully created script at: {script_path}")

Successfully created script at: /content/drive/MyDrive/AI-ML-Internship/Week-6-LLM-API-Summarizer/src/summarizer.py


In [8]:
# 1. requirements.txt banana
req_path = PROJECT_DIR / "requirements.txt"
with open(req_path, "w", encoding="utf-8") as f:
    f.write("google-genai\n")

# 2. .gitignore banana
gitignore_path = PROJECT_DIR / ".gitignore"
gitignore_content = """
# Python
__pycache__/
*.pyc

# Jupyter
.ipynb_checkpoints/

# Environment variables & Keys
.env
*.key
secrets/
credentials/
"""
with open(gitignore_path, "w", encoding="utf-8") as f:
    f.write(gitignore_content.strip())

# 3. outputs/sample_summary.txt banana
output_file = OUTPUTS_DIR / "sample_summary.txt"
sample_output_content = """INPUT:
Artificial intelligence is being adopted across many industries to automate tasks, analyze data, improve customer experiences, and support decision-making. Machine learning allows systems to learn patterns from data, while deep learning uses neural networks for more complex tasks. Generative AI can create text, images, code, and other content. Organizations must also consider privacy, security, bias, reliability, and responsible AI practices.

OUTPUT:
• AI is increasingly used for automation, data analysis, customer experiences, and decision-making.
• Machine learning, deep learning, and generative AI enable systems to learn patterns and generate different types of content.
• AI adoption requires attention to privacy, security, bias, reliability, and responsible use.
"""
with open(output_file, "w", encoding="utf-8") as f:
    f.write(sample_output_content)


In [9]:
# @title Gemini AI Text Summarizer (Interactive Input & Automated Storage)
from google import genai
from google.colab import userdata
from pathlib import Path
import os

# Check availability of IPython widgets for interactive UI
IPYTHON_WIDGETS_AVAILABLE = True
try:
    from ipywidgets import widgets
    from IPython.display import display, Markdown
except ImportError:
    IPYTHON_WIDGETS_AVAILABLE = False

# Define project directories
PROJECT_DIR = Path("/content/drive/MyDrive/AI-ML-Internship/Week-6-LLM-API-Summarizer")
OUTPUTS_DIR = PROJECT_DIR / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Initialize the Gemini API client securely
try:
    api_key = userdata.get("GEMINI_API_KEY")
except Exception:
    api_key = None

if not api_key:
    raise ValueError("GEMINI_API_KEY was not found in Colab Secrets. Please add it via the left sidebar (🔑).")

client = genai.Client(api_key=api_key)
MODEL_NAME = "gemini-3.8-flash"

def generate_and_save_summary(text):
    """
    Validates input text, sends a structured prompt to the Gemini API,
    displays the output, and saves the generated summary to the outputs directory.
    """
    text = text.strip()
    if not text:
        return "Error: The input text field cannot be empty."
    if len(text.split()) < 10:
        return "Warning: Please provide a longer text containing at least 10-20 words for an accurate summary."

    # Prompt engineering structure
    prompt = f"""
You are a professional text summarization assistant.
Your task is to summarize the provided text.

Rules:
1. Produce exactly 3 bullet points.
2. Each bullet should contain one important idea.
3. Keep each bullet concise and informative.
4. Do not add information that is not present in the text.
5. Return only the 3 bullet points.
6. Do not include an introduction or conclusion.

TEXT:{text}
"""
    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )
        summary = response.text

        # Automatically save the output to the outputs folder
        output_file_path = OUTPUTS_DIR / "generated_summary.txt"
        with open(output_file_path, "w", encoding="utf-8") as file:
            file.write(summary)

        print(f"[System Notice]: Summary successfully saved to -> {output_file_path}")
        return summary

    except Exception as error:
        return f"API Error: {error}"

# Create the interactive graphical interface
if IPYTHON_WIDGETS_AVAILABLE:
    text_input = widgets.Textarea(
        value='Artificial intelligence is being adopted across many industries to automate tasks, analyze data, improve customer experiences, and support decision-making. Machine learning allows systems to learn patterns from data, while deep learning uses neural networks for more complex problems. Generative AI has expanded these capabilities by allowing systems to create text, images, code, and other forms of content.',
        placeholder='Enter or paste your target text here...',
        description='Input Text:',
        disabled=False,
        layout=widgets.Layout(width='100%', height='150px')
    )

    button = widgets.Button(
        description='Generate & Save Summary',
        button_style='success',
        icon='check'
    )

    output_area = widgets.Output()

    def on_button_clicked(b):
        with output_area:
            output_area.clear_output()
            print("Processing request... communicating with the Gemini API.")
            summary = generate_and_save_summary(text_input.value)
            output_area.clear_output()
            display(Markdown(f"### Generated Summary:\n{summary}"))

    button.on_click(on_button_clicked)

    print("Please enter or paste your text into the text box below and click the button:")
    display(text_input, button, output_area)
else:
    # Fallback to standard console input if widgets are unavailable
    user_text = input("Please enter the text to summarize:\n")
    print("\nGenerating summary...\n")
    summary = generate_and_save_summary(user_text)
    print("\n=== SUMMARY ===\n")
    print(summary)

Please enter or paste your text into the text box below and click the button:


Textarea(value='Artificial intelligence is being adopted across many industries to automate tasks, analyze dat…

Button(button_style='success', description='Generate & Save Summary', icon='check', style=ButtonStyle())

Output()